***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'TIMS Data')
    path_main = os.path.join(path_sp, 'Data')
    path_out  = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Safety')
    path_tims = os.path.join(path_out, 'TIMS')

    
path_code    = os.path.join(path_git, 'Data', 'TIMS')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Importing

***

In [ ]:
path_in = os.path.join(path_tims, 'Bikes vs Pedestrians')
df_counties1 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Bikes.csv'      ))
df_counties2 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Pedestrians.csv'))

df_jurisdictions1 = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Bikes.csv'      ))
df_jurisdictions2 = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Pedestrians.csv'))

display(df_counties1     .head(3)); print('')
display(df_counties2     .head(3)); print('')
display(df_jurisdictions1.head(3)); print('')
display(df_jurisdictions2.head(3)); print('')


df_counties1      = df_counties1     .drop('Average', axis=1)
df_counties2      = df_counties2     .drop('Average', axis=1)
df_jurisdictions1 = df_jurisdictions1.drop('Average', axis=1)
df_jurisdictions2 = df_jurisdictions2.drop('Average', axis=1)

df_counties1      = pd.melt(df_counties1     , id_vars = ['County'        ], var_name = 'Year', value_name = 'Fatalities')
df_counties2      = pd.melt(df_counties2     , id_vars = ['County'        ], var_name = 'Year', value_name = 'Fatalities')
df_jurisdictions1 = pd.melt(df_jurisdictions1, id_vars = ['County', 'City'], var_name = 'Year', value_name = 'Fatalities')
df_jurisdictions2 = pd.melt(df_jurisdictions2, id_vars = ['County', 'City'], var_name = 'Year', value_name = 'Fatalities')

df_counties1     ['Category'] = 'Bikes'
df_counties2     ['Category'] = 'Pedestrians'
df_jurisdictions1['Category'] = 'Bikes'
df_jurisdictions2['Category'] = 'Pedestrians'

df_counties      = pd.concat([df_counties1     , df_counties2     ])
df_jurisdictions = pd.concat([df_jurisdictions1, df_jurisdictions2])

df_counties      = df_counties     [df_counties     ['County'] != 'Statewide']
df_jurisdictions = df_jurisdictions[df_jurisdictions['County'] != 'Statewide']

df_counties      = df_counties     .sort_values(['County',         'Year', 'Category'], ascending = [True,       True, True])
df_jurisdictions = df_jurisdictions.sort_values(['County', 'City', 'Year', 'Category'], ascending = [True, True, True, True])

df_counties     ['Fatalities_5 Year Average'] = df_counties     ['Fatalities'].rolling(window=5).mean()
df_jurisdictions['Fatalities_5 Year Average'] = df_jurisdictions['Fatalities'].rolling(window=5).mean()

df_counties     .loc[df_counties     ['Year'] == 2014, 'Fatalities_5 Year Average'] = np.nan
df_jurisdictions.loc[df_jurisdictions['Year'] == 2014, 'Fatalities_5 Year Average'] = np.nan

df_counties      = df_counties     .sort_values(['County',         'Year', 'Category'], ascending = [True,       False, True])
df_jurisdictions = df_jurisdictions.sort_values(['County', 'City', 'Year', 'Category'], ascending = [True, True, False, True])

df_counties      = df_counties     .set_index(['County',         'Year', 'Category']).reset_index()
df_jurisdictions = df_jurisdictions.set_index(['County', 'City', 'Year', 'Category']).reset_index()

display(df_counties.head(3)); print('')
display(df_jurisdictions.head(3))

In [ ]:
# Set Indicator
indicator_name = 'Safety_2'
plot_name = 'fatality_bikes_pedestrians'
export = False


## Importing ---

df_counties1 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Bikes.csv'      ))
df_counties2 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Pedestrians.csv'))


## Organizing ---


df_counties1 = df_counties1.drop('Average', axis=1)
df_counties2 = df_counties2.drop('Average', axis=1)

df_counties1 = pd.melt(df_counties1, id_vars = ['County'], var_name = 'Year', value_name = 'Fatalities')
df_counties2 = pd.melt(df_counties2, id_vars = ['County'], var_name = 'Year', value_name = 'Fatalities')

df_counties1     ['Category'] = 'Bikes'
df_counties2     ['Category'] = 'Pedestrians'

df_plot = pd.concat([df_counties1, df_counties2])


df_plot['County'] = df_plot['County'].str.title()

df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Sacramento', 'Placer', 'Sutter', 'Yolo', 'Yuba'])]

df_plot['Year'] = df_plot['Year'].astype(int)
df_plot['Fatalities'] = df_plot['Fatalities'].astype(int)

df_plot = df_plot.groupby(['Year', 'Category'], as_index=False)['Fatalities'].sum()
df_plot = df_plot.sort_values(['Year', 'Category'], ascending = [False, False])

df_plot = df_plot.set_index(['Year', 'Category']).reset_index()

display(df_plot.head())


## Plotting ---


color_map = {
    'Bikes': "#9DC209"
    , 'Pedestrians': "#1E90FF"
}


fig = px.bar(df_plot, y='Fatalities', x='Year'
             , color='Category'
             , color_discrete_map=color_map)


title = 'Fatalities by Counties, Bikes vs Pedestrians'
# fig.update_layout(showlegend = False)
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')


plot_agol(export=export)